In [1]:
import pandas as pd
import os
from typing import cast
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from data_preprocessing import create_train_test_val_sets, get_processed_df
from scipy.sparse import csr_matrix
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler
from sklearn.ensemble import RandomForestClassifier


In [2]:
#Create test train splits
x_mendeley, y_mendeley = get_processed_df(r"../data/raw/Mendeley Dataset.csv")
x_kaggle, y_kaggle= get_processed_df(r"../data/raw/dataset_phishing.csv")

mendeley_sets = create_train_test_val_sets(x_mendeley,y_mendeley, label_col="Label", test_size=0.2, n_splits=5)
kaggle_sets = create_train_test_val_sets(x_kaggle,y_kaggle, label_col="Label", test_size=0.2, n_splits=5)

----------Processing None Dataset----------

Class Distribution:
Label
0    0.518415
1    0.481585
Name: proportion, dtype: float64
int64

Total Missing Values: 0
No categorical features to hash
Shape After Processing: (247950, 42)
True
----------Processing None Dataset----------

Class Distribution:
Label
legitimate    0.5
phishing      0.5
Name: proportion, dtype: float64
object

Total Missing Values: 0
Shape After Processing: (11430, 32856)
True
Train/validation/test split prepared: 210757 instances for training, 37193 instances for validation, 49590 instances for testing
Stratified 5-fold CV splits created.
Train/validation/test split prepared: 9715 instances for training, 1715 instances for validation, 2286 instances for testing
Stratified 5-fold CV splits created.


### Tuning Classifiers

In [3]:
#XGBoost

def optimize_xgboost(X, y, splits) -> XGBClassifier:
    """
    Returns a trained and tuned XGBClassifier
    
    Parameters:
    X: input data
    y: target variable
    dataset: which dataset is being used to tune the XGBClassifier
    splits: A list of tuples containing the splits for CV

    Returns:
    Tuned XGBClassifier
    """
    if hasattr(X, "sparse"):
        X = csr_matrix(X.sparse.to_coo())

    scale_weights = [1.0]
    counts = y.value_counts(normalize=True)
    scale_weights.append(counts[0]/counts[1]) 
    
    params = {
        'max_depth': [4, 5, 6, 8, 10],
        'gamma': [0.1, 0.2],
        'subsample': [0.6, 0.7],
        'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.4],
        'n_estimators': [300, 500, 750, 1000, 1250],
        'scale_pos_weight': scale_weights
    }

    xgb = XGBClassifier(random_state=42)
    random_search = RandomizedSearchCV(xgb, param_distributions=params, random_state=42, cv=splits)
    random_search.fit(X, y)

    print('\n Best hyperparameters:')
    print(random_search.best_params_)

    return cast(XGBClassifier, random_search.best_estimator_)

os.makedirs("./models/phase_1", exist_ok=True)
print("Running hyperparameter tuning using Mendeley Dataset:")
xgboost_mendeley = optimize_xgboost(mendeley_sets["x_train"], mendeley_sets["y_train"], mendeley_sets["cv_splits"])
joblib.dump(xgboost_mendeley, "./models/phase_1/xgboost_mendeley_no_fs.joblib")
#{'subsample': 0.6, 'scale_pos_weight': np.float64(1.076485019261653), 'n_estimators': 750, 'max_depth': 8, 'learning_rate': 0.1, 'gamma': 0.1}

print("Running hyperparameter tuning using kaggle Dataset:")
xgboost_kaggle = optimize_xgboost(kaggle_sets["x_train"], kaggle_sets["y_train"], kaggle_sets["cv_splits"])
joblib.dump(xgboost_kaggle, "./models/phase_1/xgboost_kaggle_no_fs.joblib")
#{'subsample': 0.6, 'scale_pos_weight': np.float64(0.9997941539728283), 'n_estimators': 750, 'max_depth': 8, 'learning_rate': 0.1, 'gamma': 0.1}


Running hyperparameter tuning using Mendeley Dataset:

 Best hyperparameters:
{'subsample': 0.6, 'scale_pos_weight': 1.076485019261653, 'n_estimators': 750, 'max_depth': 8, 'learning_rate': 0.1, 'gamma': 0.1}
Running hyperparameter tuning using kaggle Dataset:

 Best hyperparameters:
{'subsample': 0.6, 'scale_pos_weight': 0.9997941539728283, 'n_estimators': 750, 'max_depth': 8, 'learning_rate': 0.1, 'gamma': 0.1}


['./models/phase_1/xgboost_kaggle_no_fs.joblib']

In [4]:
#tuning Logistic Regression
def optimize_logistic_regression(X, y, splits):
    """
    Tune Logistic Regression using cross-validation splits.

    Parameters:
    X: input features
    y: target labels
    splits: list of (train_idx, val_idx) tuples for CV

    Returns:
    best fitted pipeline
    """
    params = {
        "model__C": [0.01, 0.1, 1, 10]
    }

    pipeline = Pipeline([
        ("scaler", MaxAbsScaler()),
        ("model", LogisticRegression(
            solver="saga",
            class_weight="balanced",
            max_iter=5000,
            tol=1e-3,
            random_state=42
        ))
    ])

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=splits,
        scoring="f1",
        n_jobs=1
    )

    grid_search.fit(X, y)

    print("\nBest hyperparameters:")
    print(grid_search.best_params_)
    print("Best CV F1:", grid_search.best_score_)

    return grid_search.best_estimator_

print("Running hyperparameter tuning using Mendeley Dataset:")
logreg_mendeley = optimize_logistic_regression(
    mendeley_sets["x_train"],
    mendeley_sets["y_train"],
    mendeley_sets["cv_splits"]
)

joblib.dump(logreg_mendeley, "./models/phase_1/logreg_mendeley_no_fs.joblib")

print("Running hyperparameter tuning using kaggle Dataset:")
logreg_kaggle = optimize_logistic_regression(
    kaggle_sets["x_train"],
    kaggle_sets["y_train"],
    kaggle_sets["cv_splits"]
)

joblib.dump(logreg_kaggle, "./models/phase_1/logreg_kaggle_no_fs.joblib")

Running hyperparameter tuning using Mendeley Dataset:

Best hyperparameters:
{'model__C': 10}
Best CV F1: 0.7856266803613682
Running hyperparameter tuning using kaggle Dataset:

Best hyperparameters:
{'model__C': 10}
Best CV F1: 0.9674891254180071


['./models/phase_1/logreg_kaggle_no_fs.joblib']

In [5]:
#tuning Random Forest
param_grid = {
    'n_estimators': [100, 200, 500, 1000],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt', 'log2'],
    'class_weight': [None, 'balanced']
}
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_tuned_mendeley = RandomizedSearchCV(rf_model, param_grid, cv=mendeley_sets["cv_splits"], scoring='f1_weighted', n_jobs=-1, verbose=1)
rf_tuned_mendeley.fit(mendeley_sets['x_train'], mendeley_sets['y_train'])
print('Best params (Mendeley):', rf_tuned_mendeley.best_params_)
rf_mendeley = rf_tuned_mendeley.best_estimator_
joblib.dump(rf_mendeley, "./models/phase_1/rf_mendeley_no_fs.joblib")

rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_tuned_kaggle = RandomizedSearchCV(rf_model, param_grid, cv=kaggle_sets["cv_splits"], scoring='f1_weighted', n_jobs=-1, verbose=1)
rf_tuned_kaggle.fit(kaggle_sets['x_train'], kaggle_sets['y_train'])
print('Best params (Kaggle):', rf_tuned_kaggle.best_params_)
rf_kaggle = rf_tuned_kaggle.best_estimator_
joblib.dump(rf_kaggle, "./models/phase_1/rf_kaggle_no_fs.joblib")

Fitting 5 folds for each of 10 candidates, totalling 50 fits


/Users/ovaisazeem/Library/Python/3.9/lib/python/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best params (Mendeley): {'n_estimators': 200, 'min_samples_split': 2, 'max_features': 'sqrt', 'max_depth': None, 'class_weight': 'balanced'}
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best params (Kaggle): {'n_estimators': 100, 'min_samples_split': 2, 'max_features': 'sqrt', 'max_depth': None, 'class_weight': None}


['./models/phase_1/rf_kaggle_no_fs.joblib']

### Predicting using no feature selection

In [6]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_score, recall_score

def train_no_feature_selection(dataset, model, model_name, dataset_name):
    """
    Predict using the testset and print the confusion matrix and classification report

    Parameters:
    x_test: the test set for input variables
    y_test: the test set for the target variable
    model: the trained model
    """

    X_train = csr_matrix(dataset["x_train"])
    X_test = csr_matrix(dataset["x_test"])

    model.fit(X_train, dataset["y_train"])
    y_test_pred = model.predict(X_test)

    # Metrics
    f1 = f1_score(dataset["y_test"], y_test_pred)
    prec = precision_score(dataset["y_test"], y_test_pred)
    rec = recall_score(dataset["y_test"], y_test_pred)

    print(f"\n{model_name} - {dataset_name}")
    print(f"F1: {f1:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}")

    # Confusion Matrix
    cm = confusion_matrix(dataset["y_test"], y_test_pred)
    disp = ConfusionMatrixDisplay(cm)
    disp.plot()
    plt.title(f"{model_name} - {dataset_name}")
    plt.savefig(f"../results/phase_1/plots/{model_name}_{dataset_name}_cm.png")
    plt.close()

    return {
        "Dataset": dataset_name,
        "Model": model_name,
        "F1": f1,
        "Precision": prec,
        "Recall": rec
    }

os.makedirs("../results/phase_1", exist_ok=True)
os.makedirs("../results/phase_1/data", exist_ok=True)
os.makedirs("../results/phase_1/plots", exist_ok=True)

baseline_results = []

# XGBoost
baseline_results.append(train_no_feature_selection(mendeley_sets, xgboost_mendeley, "XGBoost", "Mendeley"))
baseline_results.append(train_no_feature_selection(kaggle_sets, xgboost_kaggle, "XGBoost", "Kaggle"))

# Logistic Regression
baseline_results.append(train_no_feature_selection(mendeley_sets, logreg_mendeley, "LogReg", "Mendeley"))
baseline_results.append(train_no_feature_selection(kaggle_sets, logreg_kaggle, "LogReg", "Kaggle"))

# Random Forest
baseline_results.append(train_no_feature_selection(mendeley_sets, rf_mendeley, "RF", "Mendeley"))
baseline_results.append(train_no_feature_selection(kaggle_sets, rf_kaggle, "RF", "Kaggle"))

df = pd.DataFrame(baseline_results)
df.to_csv("../results/phase_1/data/baseline_results.csv", index=False)

print("Saved baseline results to ../results/phase_1/data/baseline_results.csv")

for dataset in df["Dataset"].unique():
    subset = df[df["Dataset"] == dataset]

    plt.figure()
    plt.bar(subset["Model"], subset["F1"])
    plt.title(f"Baseline Model Comparison - {dataset}")
    plt.xlabel("Model")
    plt.ylabel("F1 Score")
    plt.savefig(f"../results/phase_1/plots/{dataset}_baseline_comparison.png")
    plt.close()


XGBoost - Mendeley
F1: 0.9516, Precision: 0.9613, Recall: 0.9422

XGBoost - Kaggle
F1: 0.9769, Precision: 0.9739, Recall: 0.9799

LogReg - Mendeley
F1: 0.7851, Precision: 0.8327, Recall: 0.7426

LogReg - Kaggle
F1: 0.9759, Precision: 0.9780, Recall: 0.9738

RF - Mendeley
F1: 0.9755, Precision: 0.9798, Recall: 0.9711

RF - Kaggle
F1: 0.9719, Precision: 0.9762, Recall: 0.9676
Saved baseline results to ../results/phase_1/data/baseline_results.csv


In [7]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve

def plot_all_models_curves(models, dataset, dataset_name):
    X_train = csr_matrix(dataset["x_train"])
    X_test = csr_matrix(dataset["x_test"])
    y_true = dataset["y_test"]

    # ROC
    plt.figure()
    for model, name in models:
        model.fit(X_train, dataset["y_train"])
        y_scores = model.predict_proba(X_test)[:, 1]

        fpr, tpr, _ = roc_curve(y_true, y_scores)
        roc_auc = auc(fpr, tpr)

        plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.3f})")

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve - {dataset_name}")
    plt.legend()
    plt.savefig(f"../results/phase_1/plots/{dataset_name}_roc_all.png")
    plt.close()

    # PR
    plt.figure()
    for model, name in models:
        model.fit(X_train, dataset["y_train"])
        y_scores = model.predict_proba(X_test)[:, 1]

        precision, recall, _ = precision_recall_curve(y_true, y_scores)
        plt.plot(recall, precision, label=name)

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision-Recall Curve - {dataset_name}")
    plt.legend()
    plt.savefig(f"../results/phase_1/plots/{dataset_name}_pr_all.png")
    plt.close()
    
models_mendeley = [
    (xgboost_mendeley, "XGBoost"),
    (rf_mendeley, "RF"),
    (logreg_mendeley, "LogReg")
]

models_kaggle = [
    (xgboost_kaggle, "XGBoost"),
    (rf_kaggle, "RF"),
    (logreg_kaggle, "LogReg")
]

plot_all_models_curves(models_mendeley, mendeley_sets, "Mendeley")
plot_all_models_curves(models_kaggle, kaggle_sets, "Kaggle")